In [3]:
from groq import Groq
import os, getpass

# Colab Secrets (key name: GROQ_API_KEY) -> env variable -> hidden prompt
api_key = None
try:
    from google.colab import userdata
    api_key = userdata.get("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")
except Exception:
    pass
api_key = api_key or os.getenv("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") or getpass.getpass("Enter your Groq API key: ")

client = Groq(api_key=api_key)
MODEL = "openai/gpt-oss-20b"
print("Groq connected successfully!")

Enter your Groq API key: ··········
Groq connected successfully!


In [2]:
!pip install groq requests pandas gradio

In [4]:
TUTOR_SYSTEM_PROMPT = """
You are PyBuddy, a friendly and patient Python programming tutor for ABSOLUTE BEGINNERS.

ROLE
- Teach basic Python: variables, data types, conditionals (if/elif/else),
  loops (for/while), functions, lists and simple input/output.
- If the question is not about Python/programming, politely bring the student back to Python.

HOW TO EXPLAIN
- Use simple, everyday English. Avoid jargon; if you use a technical word, explain it immediately.
- Start with a real-life analogy (e.g. "a variable is like a labelled box").
- One idea at a time, in small steps. Never assume prior knowledge.

EXAMPLES
- Always include at least one short, runnable snippet in a ```python block (under ~10 lines).
- Use friendly names (name, age, fruits).
- After each snippet explain what the lines do and show the expected output.

GUIDE, DON'T JUST ANSWER
- For homework-style problems, do NOT hand over the finished solution.
  Give hints, break the problem into steps, and ask guiding questions.
- If the student shares buggy code, point to the problem area and let them fix it.
- End EVERY reply with ONE small practice question and invite them to try it.

ADAPT TO THE STUDENT
- If they sound confused, simplify and use a new analogy.
- If they sound confident, add one small extension idea.
- Be encouraging. Mistakes are normal.

FORMAT
- Short paragraphs, code blocks, under ~250 words.
"""

def teaching_agent(user_prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": TUTOR_SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.4,
    )
    return response.choices[0].message.content

In [5]:
import json, re

RUBRIC = {
    "conceptual_accuracy":   "Python content is technically correct, nothing misleading",
    "clarity":               "Well organised, logical, easy to follow",
    "beginner_friendliness": "Simple language, jargon explained, encouraging tone",
    "example_quality":       "Correct, relevant, short code examples with explanation/output",
    "teaching_approach":     "Guides with hints/questions/practice task instead of just giving answers",
}
SCORE_KEYS = list(RUBRIC)

JUDGE_SYSTEM_PROMPT = """
You are a strict but fair Teaching Evaluator. You assess how well an AI Python tutor
answered a beginner's question.

Score each criterion with an INTEGER from 0 to 5 (0 = very poor, 3 = acceptable, 5 = excellent):
1. conceptual_accuracy      - Is the Python content correct and not misleading?
2. clarity                  - Is it clear, well structured and easy to follow?
3. beginner_friendliness    - Simple language, jargon explained, encouraging tone?
4. example_quality          - Correct, relevant, simple code examples with explanation?
5. teaching_approach        - Guides the student (hints, questions, practice task)
                              instead of just giving the answer?

Rules:
- Judge ONLY the tutor's response in the context of the student's question.
- Do not give full marks by default; justify scores with specifics.
- total_score must equal the sum of the five scores (max 25).
- Output ONLY a valid JSON object, nothing else.

JSON schema:
{
  "conceptual_accuracy": <int 0-5>,
  "clarity": <int 0-5>,
  "beginner_friendliness": <int 0-5>,
  "example_quality": <int 0-5>,
  "teaching_approach": <int 0-5>,
  "total_score": <int 0-25>,
  "feedback": "<2-3 sentences explaining the evaluation>",
  "suggestions": "<concrete suggestions for improvement>"
}
"""

def _extract_json(text):
    text = re.sub(r"```(?:json)?", "", text).strip()
    s, e = text.find("{"), text.rfind("}")
    if s == -1 or e == -1:
        raise ValueError("No JSON found in judge output")
    return json.loads(text[s:e + 1])

def _validate(result):
    for k in SCORE_KEYS:
        if k not in result:
            raise ValueError(f"Missing key: {k}")
        result[k] = max(0, min(5, int(result[k])))
    result["total_score"] = sum(result[k] for k in SCORE_KEYS)   # never trust LLM arithmetic
    result.setdefault("feedback", "")
    result.setdefault("suggestions", "")
    return result

def llm_judge(question, tutor_response, retries=2):
    user_msg = (f"STUDENT QUESTION:\n{question}\n\n"
                f"TUTOR RESPONSE TO EVALUATE:\n{tutor_response}\n\n"
                "Return the JSON evaluation now.")
    last_err = None
    for _ in range(retries + 1):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            temperature=0,
            response_format={"type": "json_object"},
        )
        try:
            return _validate(_extract_json(resp.choices[0].message.content))
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Judge failed after retries: {last_err}")

In [6]:
def run_pipeline(question):
    answer = teaching_agent(question)
    evaluation = llm_judge(question, answer)
    return answer, evaluation

def show(question):
    answer, ev = run_pipeline(question)
    print("=" * 70)
    print("STUDENT :", question)
    print("=" * 70)
    print("\n🧑‍🏫 TEACHING AGENT:\n")
    print(answer)
    print("\n" + "=" * 70)
    print("⚖️  LLM JUDGE EVALUATION")
    print("=" * 70)
    print(json.dumps(ev, indent=2))
    print(f"\nTOTAL: {ev['total_score']}/25\n")

# Take screenshots of these 3 runs for your submission
for q in ["What is a Python variable?",
          "How does a for loop work?",
          "Write me a function that checks if a number is even"]:
    show(q)

STUDENT : What is a Python variable?

🧑‍🏫 TEACHING AGENT:

A **variable** is just a name that holds a value, like a labelled box that can store something for you to use later.

Think of it like this:  
- You have a box called `name`.  
- Inside that box you put the word `"Alice"`.  
- Later you can open the box and read `"Alice"` or put something else in it.

In Python you create a variable by writing the name, an equals sign `=`, and the value you want to store.

```python
name = "Alice"          # put the string Alice into the box called name
age = 30                # put the number 30 into the box called age
height = 1.65           # put the number 1.65 into the box called height
```

**What each line does**

1. `name = "Alice"` – creates a box called `name` and stores the text `"Alice"`.
2. `age = 30` – creates a box called `age` and stores the number `30`.
3. `height = 1.65` – creates a box called `height` and stores the number `1.65`.

You can later use those variables:

```pytho

In [7]:
import pandas as pd

TEST_QUESTIONS = [
    "What is a Python variable?",
    "What are data types in Python?",
    "How does an if-else statement work?",
    "Explain for loop vs while loop",
    "What is a function and why do we use it?",
    "Write a program to find the largest of 3 numbers",   # should guide, not just solve
    "My code gives an error: print(Hello)",               # debugging
    "Who won the cricket world cup?",                     # off-topic handling
]

rows = []
for q in TEST_QUESTIONS:
    try:
        ans, ev = run_pipeline(q)
        rows.append({"question": q, **{k: ev[k] for k in SCORE_KEYS},
                     "total": ev["total_score"], "feedback": ev["feedback"]})
    except Exception as e:
        rows.append({"question": q, "feedback": f"ERROR: {e}"})

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 80)
display(df[["question"] + SCORE_KEYS + ["total"]])
print("\nAverage per criterion:")
print(df[SCORE_KEYS + ["total"]].mean().round(2))
print(f"\nOverall average: {df['total'].mean():.1f}/25")
df.to_csv("evaluation_results.csv", index=False)

,question,conceptual_accuracy,clarity,beginner_friendliness,example_quality,teaching_approach,total
0,What is a Python variable?,5,5,5,5,5,25
1,What are data types in Python?,4,5,5,5,4,23
2,How does an if-else statement work?,5,5,5,5,4,24
3,Explain for loop vs while loop,5,5,5,5,5,25
4,What is a function and why do we use it?,5,5,5,5,5,25
5,Write a program to find the largest of 3 numbers,3,5,5,4,5,22
6,My code gives an error: print(Hello),5,5,5,5,5,25
7,Who won the cricket world cup?,0,4,5,5,5,19



Average per criterion:
conceptual_accuracy       4.00
clarity                   4.88
beginner_friendliness     5.00
example_quality           4.88
teaching_approach         4.75
total                    23.50
dtype: float64

Overall average: 23.5/25


In [8]:
import gradio as gr

def ui_fn(prompt):
    if not prompt.strip():
        return "Please enter a Python question.", "{}", ""
    try:
        answer, ev = run_pipeline(prompt)
        return answer, json.dumps(ev, indent=2), f"## ⚖️ Score: {ev['total_score']}/25"
    except Exception as e:
        return f"Error: {e}", "{}", ""

app = gr.Interface(
    fn=ui_fn,
    inputs=gr.Textbox(lines=3, label="Ask PyBuddy",
                      placeholder="Example: What is a Python list?"),
    outputs=[
        gr.Markdown(label="🧑‍🏫 Teaching Agent"),
        gr.Code(language="json", label="⚖️ LLM Judge Evaluation (JSON)"),
        gr.Markdown(label="Total Score"),
    ],
    title="🐍 PyBuddy: Python Teaching Agent + LLM Judge",
    description="Learn Python basics. Every answer is scored by a second AI judge.",
)
app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://725a4f0cfbf6700313.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
